In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path("..") / "data" / "processed"

prices = pd.read_csv(PROCESSED / "prices_close.csv", index_col=0, parse_dates=True)
stocks = pd.read_csv(PROCESSED / "stock_metrics.csv", index_col=0)

# Trend and momentum (same rules as notebook 04)
sma50, sma200 = prices.rolling(50).mean().iloc[-1], prices.rolling(200).mean().iloc[-1]
last = prices.iloc[-1]
trend = pd.Series("Sideways", index=prices.columns)
trend[(last > sma50) & (sma50 > sma200)] = "Uptrend"
trend[(last < sma50) & (sma50 < sma200)] = "Downtrend"
stocks["trend"] = trend
stocks["momentum_6m_%"] = ((prices.iloc[-1] / prices.iloc[-126] - 1) * 100).round(2)

def scale(s):
    return (s - s.min()) / (s.max() - s.min()) * 100

# Four components, each on a 0-100 scale (higher = better)
stocks["performance_pts"] = scale(stocks["sharpe"])
stocks["safety_pts"] = 100 - stocks["risk_score"]
stocks["trend_pts"] = stocks["trend"].map({"Uptrend": 100, "Sideways": 50, "Downtrend": 0})
stocks["momentum_pts"] = scale(stocks["momentum_6m_%"])

stocks["total_score"] = (
    0.35 * stocks["performance_pts"]
    + 0.25 * stocks["safety_pts"]
    + 0.25 * stocks["trend_pts"]
    + 0.15 * stocks["momentum_pts"]
).round(1)

stocks[["sector", "sharpe", "risk_score", "trend", "momentum_6m_%", "total_score"]] \
    .sort_values("total_score", ascending=False)

,sector,sharpe,risk_score,trend,momentum_6m_%,total_score
ICICIBANK,Banking,0.36,18.5,Uptrend,4.60,80.6
BHARTIARTL,Infra/Others,0.78,14.7,Sideways,4.44,79.0
SUNPHARMA,Pharma,0.51,4.5,Sideways,5.25,75.4
BAJAJ_AUTO,Auto,0.71,51.4,Sideways,7.96,69.5
KOTAKBANK,Banking,-0.21,35.4,Uptrend,16.83,67.6
CIPLA,Pharma,0.08,21.2,Sideways,7.80,62.0
NESTLEIND,FMCG,-0.02,5.5,Sideways,-1.09,60.1
DRREDDY,Pharma,-0.07,15.1,Sideways,0.41,57.1
AXISBANK,Banking,0.11,40.7,Sideways,-5.01,52.8
MandM,Auto,0.73,55.0,Downtrend,-7.77,50.4


In [3]:
def label(score):
    if score >= 60:
        return "Buy"
    if score < 35:
        return "Sell"
    return "Hold"

def reason(r):
    perf = ("strong long-term returns" if r["sharpe"] >= 0.5
            else "weak long-term returns" if r["sharpe"] < 0
            else "moderate long-term returns")
    return f"{perf}, {str(r['risk_level']).lower()} risk, {r['trend'].lower()}"

stocks["recommendation"] = stocks["total_score"].apply(label)
stocks["reason"] = stocks.apply(reason, axis=1)

stocks.to_csv(PROCESSED / "recommendations.csv")

print(stocks["recommendation"].value_counts())
stocks[["sector", "total_score", "recommendation", "reason"]] \
    .sort_values("total_score", ascending=False)

recommendation
Hold    12
Sell     9
Buy      7
Name: count, dtype: int64


,sector,total_score,recommendation,reason
ICICIBANK,Banking,80.6,Buy,"moderate long-term returns, low risk, uptrend"
BHARTIARTL,Infra/Others,79.0,Buy,"strong long-term returns, low risk, sideways"
SUNPHARMA,Pharma,75.4,Buy,"strong long-term returns, low risk, sideways"
BAJAJ_AUTO,Auto,69.5,Buy,"strong long-term returns, medium risk, sideways"
KOTAKBANK,Banking,67.6,Buy,"weak long-term returns, medium risk, uptrend"
CIPLA,Pharma,62.0,Buy,"moderate long-term returns, low risk, sideways"
NESTLEIND,FMCG,60.1,Buy,"weak long-term returns, low risk, sideways"
DRREDDY,Pharma,57.1,Hold,"weak long-term returns, low risk, sideways"
AXISBANK,Banking,52.8,Hold,"moderate long-term returns, medium risk, sideways"
MandM,Auto,50.4,Hold,"strong long-term returns, medium risk, downtrend"
